# Fine-tune Llama 3.2 3B for Tool Calling
**Paper**: *Adaptation of Agentic AI: A Survey of Post-Training, Memory, and Skills* (arXiv:2512.16301)

This notebook reproduces the **A1 paradigm** (supervised fine-tuning of an agent) from the survey:
- **Model**: `meta-llama/Llama-3.2-3B-Instruct` with LoRA via unsloth
- **Dataset**: `Salesforce/xlam-function-calling-60k`
- **Goal**: Teach the model to output valid tool-call JSON

Each iteration, `autoresearch/config.yaml` is updated by `run_loop.py` with new hyperparams.
Run all cells, then paste the final metrics JSON back into the terminal running `run_loop.py`.

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
# Run this once per Colab session
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets huggingface_hub transformers pyyaml
print('Done.')

In [ ]:
# ── Cell 2: Clone repo & load config ──────────────────────────────────────────
# Replace with your actual GitHub repo URL
GITHUB_REPO = "https://github.com/YOUR_GITHUB_USERNAME/agentic_ai_adaptation.git"

import os
if not os.path.exists("agentic_ai_adaptation"):
    !git clone {GITHUB_REPO}
else:
    !git -C agentic_ai_adaptation pull

import yaml
with open("agentic_ai_adaptation/autoresearch/config.yaml") as f:
    config = yaml.safe_load(f)

print(f"Iteration   : {config['iteration']}")
print(f"lora_r      : {config['lora_r']}")
print(f"lr          : {config['learning_rate']}")
print(f"max_steps   : {config['max_steps']}")
print(f"adapter_repo: {config['adapter_repo']}")
print(f"notes       : {config['notes']}")

In [ ]:
# ── Cell 3: HuggingFace login ──────────────────────────────────────────────────
from huggingface_hub import login
from google.colab import userdata

# Store your HF token in Colab Secrets as HF_TOKEN
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print('Logged in to HuggingFace.')

In [ ]:
# ── Cell 4: Load base model with unsloth (4-bit QLoRA) ────────────────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=config["base_model"],
    max_seq_length=config["max_seq_length"],
    dtype=None,          # auto-detect bf16/fp16
    load_in_4bit=True,   # QLoRA
)
print('Base model loaded.')

In [ ]:
# ── Cell 5: Apply LoRA adapters ────────────────────────────────────────────────
model = FastLanguageModel.get_peft_model(
    model,
    r=config["lora_r"],
    target_modules=config["lora_target_modules"],
    lora_alpha=config["lora_alpha"],
    lora_dropout=config["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=config.get("seed", 42),
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

In [ ]:
# ── Cell 6: Load & format dataset ─────────────────────────────────────────────
import json
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are a helpful assistant with access to tools. "
    "When you need to use a tool, respond ONLY with a JSON object in this exact format:\n"
    '{"name": "tool_name", "arguments": {"param": "value"}}\n'
    "Do not add any explanation before or after the JSON."
)

full_dataset = load_dataset(config["dataset_name"], split="train")
n_total = len(full_dataset)
train_end = int(n_total * config["train_split_ratio"])
max_samples = min(config.get("max_train_samples", 5000), train_end)

# Shuffle before subsetting for diversity
train_data = full_dataset.select(range(train_end)).shuffle(seed=config.get("seed", 42))
train_data = train_data.select(range(max_samples))
print(f"Training on {len(train_data)} samples (out of {train_end} available)")


def format_row(row):
    tools = row.get("tools", "[]")
    if isinstance(tools, str):
        tools = json.loads(tools)
    answers = row.get("answers", "[]")
    if isinstance(answers, str):
        answers = json.loads(answers)
    answer_str = json.dumps(answers[0]) if answers else ""

    return {
        "text": (
            "<|begin_of_text|>"
            "<|start_header_id|>system<|end_header_id|>\n"
            f"{SYSTEM_PROMPT}\n\n"
            f"Available tools:\n{json.dumps(tools, indent=2)}"
            "<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n"
            f"{row['query']}"
            "<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n"
            f"{answer_str}"
            "<|eot_id|>"
        )
    }


train_data = train_data.map(format_row, remove_columns=train_data.column_names)
print("Sample:")
print(train_data[0]["text"][:400], "...")

In [ ]:
# ── Cell 7: Train with SFTTrainer ─────────────────────────────────────────────
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    dataset_text_field="text",
    max_seq_length=config["max_seq_length"],
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=config["per_device_train_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],
        warmup_ratio=config["warmup_ratio"],
        max_steps=config["max_steps"],
        learning_rate=config["learning_rate"],
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        lr_scheduler_type=config["lr_scheduler_type"],
        seed=config.get("seed", 42),
        output_dir="outputs",
        report_to="none",
    ),
)

trainer_stats = trainer.train()
print(f"Training complete. Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")

In [ ]:
# ── Cell 8: Evaluate on held-out split ────────────────────────────────────────
import torch

FastLanguageModel.for_inference(model)

eval_start = int(n_total * config["train_split_ratio"])
eval_samples = config.get("eval_samples", 200)
eval_data = full_dataset.select(range(eval_start, min(eval_start + eval_samples, n_total)))
print(f"Evaluating on {len(eval_data)} samples...")

fmt_ok = ans_ok = total = 0

for i, row in enumerate(eval_data):
    query = row.get("query", "").strip()
    tools = row.get("tools", "[]")
    answers = row.get("answers", "[]")
    if isinstance(tools, str):
        tools = json.loads(tools)
    if isinstance(answers, str):
        answers = json.loads(answers)
    if not query or not answers:
        continue

    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        f"{SYSTEM_PROMPT}\n\nAvailable tools:\n{json.dumps(tools, indent=2)}"
        "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{query}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=128,
            do_sample=False, pad_token_id=tokenizer.eos_token_id
        )
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    try:
        parsed = json.loads(raw)
        fmt = "name" in parsed and "arguments" in parsed
    except Exception:
        fmt = False
        parsed = {}

    ans = fmt and any(a.get("name") == parsed.get("name") for a in answers)
    fmt_ok += int(fmt)
    ans_ok += int(ans)
    total += 1

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(eval_data)}  format={fmt_ok/total:.1%}  answer={ans_ok/total:.1%}")

fmt_acc = fmt_ok / total
ans_acc = ans_ok / total
combined = 0.5 * fmt_acc + 0.5 * ans_acc

print(f"\nFormat accuracy : {fmt_acc:.2%}")
print(f"Answer accuracy : {ans_acc:.2%}")
print(f"Combined        : {combined:.2%}")

In [ ]:
# ── Cell 9: Push adapter to HuggingFace Hub ───────────────────────────────────
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
model.push_to_hub(config["adapter_repo"])
tokenizer.push_to_hub(config["adapter_repo"])
print(f"Adapter pushed to: https://huggingface.co/{config['adapter_repo']}")

In [ ]:
# ── Cell 10: Print metrics JSON for run_loop.py ───────────────────────────────
# Copy everything between the dashes and paste into the run_loop.py terminal prompt
from datetime import datetime
import json

metrics = {
    "iteration": config["iteration"],
    "adapter_repo": config["adapter_repo"],
    "lora_r": config["lora_r"],
    "learning_rate": config["learning_rate"],
    "max_steps": config["max_steps"],
    "format_accuracy": round(fmt_acc, 4),
    "answer_accuracy": round(ans_acc, 4),
    "combined_accuracy": round(combined, 4),
    "eval_samples": total,
    "notes": config.get("notes", ""),
    "timestamp": datetime.now().isoformat(),
}

print("\n" + "─"*60)
print("COPY THIS JSON INTO run_loop.py:")
print("─"*60)
print(json.dumps(metrics))
print("─"*60)